# Slerp: Interpolación Esférica Lineal en Computación Gráfica

Autor: Julian G. — Estudiante de Sistemas y Computación, Universidad Nacional de Colombia

In [1]:
import importlib.util
import subprocess
import sys

In [2]:
missing = [pkg for pkg in ("manim", "numpy", "scipy", "matplotlib") if importlib.util.find_spec(pkg) is None]
if missing:
# manim-ce is the package distributed as 'manim' when imported
    subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "manim-ce", "numpy", "scipy", "matplotlib", "--quiet"]
    )

import os
from pathlib import Path

import numpy as np
import scipy
import scipy.spatial.transform as st
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Video, HTML

In [3]:
try:
    from manim import config
    config.pixel_height = 480          # 480p
    config.pixel_width = 854           # 16:9
    config.frame_rate = 15             # fps bajo para velocidad
    config.media_dir = "media"         # carpeta de salida
    config.preview = False             # no abrir ventana externa
    config.write_to_movie = True
    config.disable_caching = True

except Exception:
    pass

In [4]:
BLUE = "#2a78d6"
YELLOW = "#eda100"
GREEN = "#1baf7a"
RED = "#e34948"
WHITE = "#ffffff"

In [5]:
plt.style.use("dark_background")
plt.rcParams.update({
"axes.grid": True,
"grid.alpha": 0.25,
"axes.labelcolor": "#c3c2b7",
"xtick.color": "#c3c2b7",
"ytick.color": "#c3c2b7",
"text.color": "#ffffff",
"figure.facecolor": "#0d0d0d",
"axes.facecolor": "#0d0d0d",
})

def encontrar_video(nombre_escena):
    """Encuentra el MP4 generado por manim sin depender de una ruta fija."""
    base = Path("media") / "videos" / nombre_escena
    matches = sorted(base.glob("/.mp4")) if base.exists() else []
    if not matches:
        raise FileNotFoundError(f"No se encontró video para la escena {nombre_escena!r}")
    return str(matches[0])

def mostrar_video(path):
    """Reproduce un video MP4 dentro del notebook."""
    return Video(path, embed=True, html_attributes="width=520")

print("Entorno listo.")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.version}")
print(f"SciPy: {scipy.version}")
print(f"Matplotlib: {matplotlib.version}")
try:
    import manim
    print(f"manim-ce: {manim.version}")
except Exception as exc:
    print(f"manim-ce: no disponible ({exc})")
    

Entorno listo.
Python: 3.14.2
NumPy: <module 'numpy.version' from 'c:\\Users\\JulianG\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\numpy\\version.py'>
SciPy: <module 'scipy.version' from 'c:\\Users\\JulianG\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\scipy\\version.py'>


AttributeError: module 'matplotlib' has no attribute 'version'

## 1. Motivación

¿Por qué la interpolación lineal (lerp) falla para rotaciones?

Imagina dos orientaciones representadas como puntos en una esfera: $S^3$ para cuaterniones unitarios y $S^2$ para ejes-ángulo.
Si interpolamos linealmente entre ellos (lerp), los puntos intermedios caen dentro de la esfera (norma menor que 1).
Al renormalizar (nlerp) forzamos la norma a 1, pero la velocidad angular no es constante: el punto se mueve más rápido cerca de los extremos y más lento en el medio.

En la siguiente animación vemos esto en 2D (círculo unitario $S^1$): el punto rojo (lerp) se hunde dentro del círculo, mientras el verde (slerp) se mantiene en el arco y avanza a velocidad constante.

In [ ]:
from manim import *

class LerpVsSlerpS1(Scene):
    def construct(self):
        self.camera.background_color = "#0d0d0d"


        # Círculo unitario
        circle = Circle(radius=2, color=WHITE, stroke_width=2)
        self.add(circle)

        # Dos puntos en el círculo: q0 y q1 separados 120 grados
        angle0 = 0.0
        angle1 = np.deg2rad(120)
        q0 = np.array([np.cos(angle0), np.sin(angle0), 0.0])
        q1 = np.array([np.cos(angle1), np.sin(angle1), 0.0])

        dot_q0 = Dot(point=q0 * 2, color=BLUE, radius=0.10)
        dot_q1 = Dot(point=q1 * 2, color=YELLOW, radius=0.10)
        label_q0 = MathTex("q_0").next_to(dot_q0, LEFT + DOWN)
        label_q1 = MathTex("q_1").next_to(dot_q1, RIGHT + UP)
        self.add(dot_q0, dot_q1, label_q0, label_q1)

        # Puntos interpolados
        lerp_dot = Dot(point=ORIGIN, color=RED, radius=0.10)
        slerp_dot = Dot(point=ORIGIN, color=GREEN, radius=0.10)
        lerp_label = MathTex(r"\text{lerp}").next_to(lerp_dot, DOWN)
        slerp_label = MathTex(r"\text{slerp}").next_to(slerp_dot, DOWN)
        self.add(lerp_dot, slerp_dot, lerp_label, slerp_label)

        # Medidor de norma del punto lerp
        norm_label = MathTex(r"\lVert q(t) \rVert =").to_edge(UP)
        norm_value = DecimalNumber(0, num_decimal_places=3).next_to(norm_label, RIGHT)
        self.add(norm_label, norm_value)

        # Camino slerp: arco punteado
        arc = ArcBetweenPoints(q0 * 2, q1 * 2, color=GREEN, stroke_width=2)
        self.add(arc)

        # Actualizadores: ciclo de 3 segundos
        def update_lerp(mob, dt):
            alpha = min(self.time / 3.0, 1.0)
            lerp = (1 - alpha) * q0 + alpha * q1
            mob.move_to(lerp * 2)
            norm_value.set_value(np.linalg.norm(lerp))

        def update_slerp(mob, dt):
            alpha = min(self.time / 3.0, 1.0)
            omega = np.arccos(np.clip(np.dot(q0, q1), -1, 1))
            if np.abs(omega) < 1e-6:
                slerp = q0
            else:
                slerp = (
                    np.sin((1 - alpha) * omega) / np.sin(omega)
                ) * q0 + (
                    np.sin(alpha * omega) / np.sin(omega)
                ) * q1
            mob.move_to(slerp * 2)

        lerp_dot.add_updater(update_lerp)
        slerp_dot.add_updater(update_slerp)

        self.wait(3)
        lerp_dot.remove_updater(update_lerp)
        slerp_dot.remove_updater(update_slerp)

In [ ]:
scene = LerpVsSlerpS1()
scene.render()
mostrar_video(encontrar_video("LerpVsSlerpS1"))

## 2. Desarrollo matemático riguroso

2.1 Cuaterniones unitarios como puntos en $S^3$
Un cuaternión unitario
$$
q = w + x,\mathbf{i} + y,\mathbf{j} + z,\mathbf{k}
$$
cumple
$$
\lVert q\rVert^2 = w^2 + x^2 + y^2 + z^2 = 1.
$$
Por tanto, vive en la esfera 3-dimensional $S^3 \subset \mathbb{R}^4$.

Interpolar entre dos orientaciones $q_0$ y $q_1$ significa construir un camino
$$
q(t):[0,1]\longrightarrow S^3
$$
que no salga de la esfera.

2.2 Planteamiento del problema
Buscamos una función $q(t)$ que satisfaga simultáneamente:

Norma unitaria: $\lVert q(t)\rVert = 1$ para todo $t\in[0,1]$.
Velocidad angular constante: $\frac{d}{dt}\arg q(t)=\text{constante}$.
Condiciones de borde: $q(0)=q_0$ y $q(1)=q_1$.
La respuesta es la interpolación esférica lineal (spherical linear interpolation, slerp). En las celdas siguientes la derivamos desde cero, partiendo de estas tres exigencias.

### 2.3 Derivación de la fórmula de slerp

Sea
$$
\Omega = \arccos(q_0\cdot q_1)
$$
el ángulo entre $q_0$ y $q_1$ en $S^3$. Como ambos son unitarios, el plano generado por ellos corta a $S^3$ en un círculo máximo. Trabajemos en una base ortonormal de ese plano:

$$
e_1 = q_0,
\qquad
e_2 =
\frac{q_1 - (q_0\cdot q_1),q_0}
{\lVert q_1 - (q_0\cdot q_1),q_0\|}.
$$

Cualquier punto del arco mayor puede escribirse como
$$
q(t) = \cos(\theta(t)),e_1 + \sin(\theta(t)),e_2.
$$

La condición de velocidad angular constante impone $\theta(t)=\Omega t$. Sustituyendo $e_1$ y $e_2$:

$$
\begin{aligned}
q(t)
&= \cos(\Omega t)\,q_0

\sin(\Omega t)\ &\qquad\times\frac{q_1 - (q_0\cdot q_1)\,q_0}{\sin\Omega} \[4pt] &= \frac{\sin((1-t)\Omega)}{\sin\Omega}\,q_0
\frac{\sin(t\Omega)}{\sin\Omega}\,q_1. \end{aligned} $$
Esta es la fórmula de slerp:

$$ \boxed{ \operatorname{slerp}(q_0,q_1;t)
\frac{\sin((1-t)\Omega)}{\sin\Omega}\,q_0
+\frac{\sin(t\Omega)}{\sin\Omega}\,q_1
}
$$

2.4 Verificación de las propiedades exigidas
Norma unitaria. Al estar escrita como combinación de una base ortonormal con coeficientes $\cos(\Omega t)$ y $\sin(\Omega t)$, se tiene $\lVert q(t)\rVert^2=\cos^2(\Omega t)+\sin^2(\Omega t)=1$.
Velocidad angular constante. Derivando, $\dot q(t)$ es ortogonal a $q(t)$ y $\lVert\dot q(t)\rVert=\Omega$, independiente de $t$.
Condiciones de borde. En $t=0$ queda $q(0)=q_0$; en $t=1$, $q(1)=q_1$. """ ) )

In [ ]:
from manim import *

class PlanoGeneradoYDerivacion(Scene):
    def construct(self):
        self.camera.background_color = "#0d0d0d"
        # Representación 2D del plano generado por q0 y q1.
        # En S^3 esto es un círculo máximo; en 2D lo vemos sin distorsión.
        angle0 = 0.0
        angle1 = np.deg2rad(100)
        q0 = np.array([np.cos(angle0), np.sin(angle0), 0.0])
        q1 = np.array([np.cos(angle1), np.sin(angle1), 0.0])

        scale = 2.0
        v0 = q0 * scale
        v1 = q1 * scale

        dot0 = Dot(point=v0, color=BLUE, radius=0.10)
        dot1 = Dot(point=v1, color=YELLOW, radius=0.10)
        label0 = MathTex("q_0").next_to(dot0, LEFT)
        label1 = MathTex("q_1").next_to(dot1, RIGHT)
        self.add(dot0, dot1, label0, label1)

        # e1 = q0
        e1_arrow = Arrow(ORIGIN, v0, color=BLUE, buff=0, max_tip_length_to_length_ratio=0.2)
        e1_label = MathTex("e_1 = q_0").next_to(e1_arrow.get_end(), UP + LEFT)
        self.add(e1_arrow, e1_label)

        # e2 = componente ortogonal de q1 respecto a q0
        c = np.dot(q0, q1)  # cos(Omega)
        v_perp = q1 - c * q0
        if np.linalg.norm(v_perp) > 1e-6:
            e2 = v_perp / np.linalg.norm(v_perp)
        else:
            e2 = np.array([0.0, 1.0, 0.0])
        e2_arrow = Arrow(ORIGIN, e2 * scale, color=GREEN, buff=0, max_tip_length_to_length_ratio=0.2)
        e2_label = MathTex("e_2 \\\\propto q_1 - (q_0\\cdot q_1)q_0").next_to(e2_arrow.get_end(), UP)
        self.add(e2_arrow, e2_label)

        # Arco que muestra el ángulo Omega
        arc = ArcBetweenPoints(v0, v1, color=WHITE, stroke_width=2)
        omega_label = MathTex(r"\Omega").next_to(arc.get_center(), UP + LEFT)
        self.add(arc, omega_label)

        # q(t) gira a velocidad angular constante Omega*t
        qt_dot = Dot(point=ORIGIN, color=RED, radius=0.10)
        qt_label = MathTex("q(t)").next_to(qt_dot, DOWN)
        self.add(qt_dot, qt_label)

        def update_qt(mob, dt):
            alpha = min(self.time / 3.0, 1.0)
            theta = alpha * np.arccos(np.clip(c, -1, 1))
            qt = np.cos(theta) * v0 + np.sin(theta) * (e2 * scale)
            mob.move_to(qt)

        qt_dot.add_updater(update_qt)
        self.wait(3)
        qt_dot.remove_updater(update_qt)

In [ ]:
scene = PlanoGeneradoYDerivacion()
scene.render()
mostrar_video(encontrar_video("PlanoGeneradoYDerivacion"))

### 2.5 Verificación de las tres propiedades

Las tres condiciones del planteamiento quedan satisfechas por construcción:

Norma unitaria: identidad trigonométrica $\cos^2+\sin^2=1$.
Velocidad angular constante: $\lVert\dot q(t)\rVert=\Omega$.
Condiciones de borde: $q(0)=q_0$, $q(1)=q_1$.
2.6 Caso degenerado $\Omega\to 0$
Cuando $q_0$ y $q_1$ son casi paralelos, $\sin\Omega\approx 0$ y la fórmula se vuelve numéricamente inestable. En ese régimen, el arco y la cuerda casi coinciden, por lo que es seguro recurrir a nlerp (lerp + renormalización) como fallback.

2.7 Doble cobertura y corrección
Los cuaterniones $q$ y $-q$ representan la misma rotación. Si $q_0\cdot q_1<0$, el arco elegido por la fórmula es el mayor (más de $180^\circ$). Para tomar siempre el arco menor, hacemos:

$$
\text{si } q_0\cdot q_1<0,\quad q_1 \leftarrow -q_1.
$$

Esta corrección es imprescindible en aplicaciones reales: evita giros largos e inesperados cuando las orientaciones están cerca en $SO(3)$ pero lejos en $S^3$.

In [ ]:
from manim import *

class VelocidadAngular(Scene):
    def construct(self):
        self.camera.background_color = "#0d0d0d"
        # Ejes
        axes = Axes(
            x_range=[0, 1, 0.2],
            y_range=[0, 1.5, 0.2],
            x_length=6,
            y_length=4,
            axis_config={"color": WHITE, "stroke_width": 1},
            tips=False,
        )
        axes_labels = axes.get_axis_labels(x_label="t", y_label=r"\theta(t)")
        self.add(axes, axes_labels)

        # Ángulo entre q0 y q1
        Omega = np.deg2rad(120)
        c = np.cos(Omega)

        # theta(t) para slerp y nlerp
        def theta_slerp(t):
            return Omega * t

        def theta_nlerp(t):
            num = (1 - t) + t * c
            den = np.sqrt((1 - t) ** 2 + t ** 2 + 2 * t * (1 - t) * c)
            return np.arccos(np.clip(num / den, -1, 1))

        # Curvas
        slerp_curve = axes.plot(theta_slerp, x_range=[0, 1], color=GREEN, stroke_width=2)
        nlerp_curve = axes.plot(theta_nlerp, x_range=[0, 1], color=RED, stroke_width=2)
        slerp_label = axes.get_graph_label(slerp_curve, label=r"\text{slerp}", x_val=0.8, direction=UP)
        nlerp_label = axes.get_graph_label(nlerp_curve, label=r"\text{nlerp}", x_val=0.8, direction=DOWN)
        self.add(slerp_curve, nlerp_curve, slerp_label, nlerp_label)

        # Puntos móviles
        dot_s = Dot(point=axes.c2p(0, theta_slerp(0)), color=GREEN, radius=0.08)
        dot_n = Dot(point=axes.c2p(0, theta_nlerp(0)), color=RED, radius=0.08)
        self.add(dot_s, dot_n)

        def update_dots(mob, dt):
            alpha = min(self.time / 4.0, 1.0)
            t = alpha
            dot_s.move_to(axes.c2p(t, theta_slerp(t)))
            dot_n.move_to(axes.c2p(t, theta_nlerp(t)))

        dot_s.add_updater(update_dots)
        dot_n.add_updater(update_dots)

        self.wait(4)
        dot_s.remove_updater(update_dots)
        dot_n.remove_updater(update_dots)

In [ ]:
scene = VelocidadAngular()
scene.render()
mostrar_video(encontrar_video("VelocidadAngular"))

## 3. Aplicación práctica (código funcional)

Ahora pasamos de la teoría a una implementación lista para usar en gráficos por computadora. Construiremos:

slerp desde cero en numpy, con fallback numérico y corrección de doble cobertura.
Validación contra scipy.spatial.transform.Slerp.
Un benchmark de velocidad angular real frente a nlerp.
Una animación 3D de una flecha que interpola dos orientaciones. """ ) )

3.1 Implementación de slerp desde cero (numpy)

In [ ]:
import numpy as np

def slerp(q0, q1, t, epsilon=1e-6):
    r"""
    Interpolación esférica lineal entre dos cuaterniones unitarios.


    Parámetros
    ----------
    q0, q1 : array_like, forma (4,)
        Cuaterniones en orden (w, x, y, z). Se normalizan internamente.
    t : float o array_like
        Parámetro de interpolación en [0, 1].
    epsilon : float
        Umbral para usar nlerp cuando el ángulo es casi cero.

    Devuelve
    --------
    ndarray
        Cuaternión interpolado. Si t es vectorial, la forma es (..., 4).
    """
    q0 = np.asarray(q0, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    q0 = q0 / np.linalg.norm(q0)
    q1 = q1 / np.linalg.norm(q1)

    # Producto punto y corrección de doble cobertura
    dot = float(np.clip(np.dot(q0, q1), -1.0, 1.0))
    if dot < 0:
        q1 = -q1
        dot = -dot

    # Caso casi paralelo: fallback a nlerp
    if dot > 1 - epsilon:
        if np.ndim(t) == 0:
            result = (1 - t) * q0 + t * q1
            return result / np.linalg.norm(result)
        result = (1 - t)[..., None] * q0 + t[..., None] * q1
        return result / np.linalg.norm(result, axis=-1, keepdims=True)

    # Fórmula de slerp
    omega = np.arccos(dot)
    sin_omega = np.sin(omega)
    coeff0 = np.sin((1 - t) * omega) / sin_omega
    coeff1 = np.sin(t * omega) / sin_omega
    if np.ndim(t):
        coeff0 = coeff0[..., None]
        coeff1 = coeff1[..., None]
    return coeff0 * q0 + coeff1 * q1

def nlerp(q0, q1, t):
    r"""Linear interpolation + renormalización (fallback y referencia)."""
    q0 = np.asarray(q0, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    q0 = q0 / np.linalg.norm(q0)
    q1 = q1 / np.linalg.norm(q1)
    if np.ndim(t) == 0:
        result = (1 - t) * q0 + t * q1
        return result / np.linalg.norm(result)
    result = (1 - t)[..., None] * q0 + t[..., None] * q1
    return result / np.linalg.norm(result, axis=-1, keepdims=True)

In [ ]:
if __name__ == "__main__":
    q0 = [1, 0, 0, 0]
    q1 = [0, 1, 0, 0]  # 90 grados alrededor del eje z
    ts = np.linspace(0, 1, 5)
    print("slerp(q0, q1, t):")
    for ti in ts:
        print(f"  t={ti:.2f} -> {slerp(q0, q1, ti)}")

3.2 Validación frente a scipy.spatial.transform.Slerp

In [ ]:
import numpy as np
import scipy.spatial.transform as st

def compare_slerp(num_tests=200):
    max_err = 0.0
    worst_case = None
    for _ in range(num_tests):
    # Cuaterniones unitarios aleatorios en orden (w, x, y, z)
    q0 = np.random.randn(4)
    q0 = q0 / np.linalg.norm(q0)
    q1 = np.random.randn(4)
    q1 = q1 / np.linalg.norm(q1)
    # Alinear signos para comparar siempre el arco menor
        if np.dot(q0, q1) < 0:
            q1 = -q1

    # SciPy usa el orden (x, y, z, w)
    key_rot = st.Rotation.from_quat([q0[[1, 2, 3, 0]], q1[[1, 2, 3, 0]]])
    slerp_scipy = st.Slerp([0, 1], key_rot)

    t_vals = np.linspace(0, 1, 50)
    q_scipy_xyzw = slerp_scipy(t_vals)
    q_scipy_wxyz = np.roll(q_scipy_xyzw, shift=1, axis=1)  # -> (w, x, y, z)

    q_ours = np.array([slerp(q0, q1, t) for t in t_vals])
    err = np.max(np.linalg.norm(q_ours - q_scipy_wxyz, axis=1))
    if err > max_err:
        max_err = err
        worst_case = (q0, q1, err)

    print(f"Máximo error encontrado en {num_tests} pruebas: {max_err:.2e}")
    if worst_case is not None:
        print(f"Peor caso: error {worst_case[2]:.2e}")
    return max_err

err = compare_slerp(num_tests=200)
assert err < 1e-10, "Error demasiado grande"
print("Validación superada.")


3.3 Benchmark: variación de velocidad angular

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def angular_velocity_error(q0, q1, t_vals, method="slerp"):
r"""
Desviación relativa de la velocidad angular instantánea aproximada
por diferencias finitas.
"""
q0 = np.asarray(q0, dtype=float)
q1 = np.asarray(q1, dtype=float)
q0 = q0 / np.linalg.norm(q0)
q1 = q1 / np.linalg.norm(q1)
qs = np.array([slerp(q0, q1, t) for t in t_vals] if method == "slerp"
              else [nlerp(q0, q1, t) for t in t_vals])

dots = np.clip(np.sum(qs * q0, axis=1), -1, 1)
thetas = np.arccos(dots)
dt = t_vals[1] - t_vals[0]
omegas = np.diff(thetas) / dt
Omega_true = np.arccos(np.clip(np.dot(q0, q1), -1, 1))
rel_error = np.std(omegas) / np.abs(Omega_true) if np.abs(Omega_true) > 1e-6 else np.std(omegas)
return rel_error, thetas, omegas

Parámetros de prueba: rotación de 90 grados

In [ ]:
q0 = [1, 0, 0, 0]
q1 = [0, 0, 1, 0]
t_vals = np.linspace(0, 1, 101)

rel_err_slerp, theta_s, omega_s = angular_velocity_error(q0, q1, t_vals, method="slerp")
rel_err_nlerp, theta_n, omega_n = angular_velocity_error(q0, q1, t_vals, method="nlerp")

print("Desviación relativa de velocidad angular:")
print(f"  slerp: {rel_err_slerp:.2e}")
print(f"  nlerp: {rel_err_nlerp:.2e}")

Gráfica de theta(t)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_vals, theta_s, label="slerp", color=GREEN, linewidth=2)
plt.plot(t_vals, theta_n, label="nlerp", color=RED, linewidth=2)
plt.xlabel("t")
plt.ylabel(r"$\theta(t)$ [rad]")
plt.title("Ángulo acumulado vs t")
plt.legend()
plt.tight_layout()
plt.show()

Gráfica de velocidad angular instantánea

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(t_vals[:-1], omega_s, label="slerp", color=GREEN, linewidth=2)
plt.plot(t_vals[:-1], omega_n, label="nlerp", color=RED, linewidth=2)
plt.xlabel("t")
plt.ylabel(r"$d\theta/dt$ [rad/unidad]")
plt.title("Velocidad angular instantánea")
plt.legend()
plt.tight_layout()
plt.show()

3.4 Ejemplo 3D: interpolación de la orientación de una flecha

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def quat_to_rotmat(q):
r"""Convierte cuaternión (w, x, y, z) a matriz de rotación 3x3."""
w, x, y, z = q
n = np.dot(q, q)
if n < 1e-8:
return np.eye(3)
s = 2.0 / n
xs, ys, zs = x * s, y * s, z * s
wx, wy, wz = w * xs, w * ys, w * zs
xx, xy, xz = x * xs, x * ys, x * zs
yy, yz, zz = y * ys, y * ys, z * zs
return np.array([
[1 - (yy + zz), xy - wz, xz + wy],
[xy + wz, 1 - (xx + zz), yz - wx],
[xz - wy, yz + wx, 1 - (xx + yy)]
])

def arrow_points(R, scale=1.0):
r"""Puntos de una flecha (eje local +Z) transformada por R."""
base = np.array([
[0, 0, 0],
[0, 0, scale],
[-scale / 10, 0, scale * 9 / 10],
[ scale / 10, 0, scale * 9 / 10],
[0, -scale / 10, scale * 9 / 10],
[0,  scale / 10, scale * 9 / 10],
])
return (R @ base.T).T

def update_arrow(arrow_obj, R):
pts = arrow_points(R, scale=1.0)
arrow_obj.set_data(pts[:, 0], pts[:, 1])
arrow_obj.set_3d_properties(pts[:, 2])

Orientaciones inicial y final

In [ ]:
q0 = [1, 0, 0, 0]  # identidad
axis = np.array([1, 1, 1], dtype=float)
axis = axis / np.linalg.norm(axis)
angle = np.deg2rad(120)
q1 = [
np.cos(angle / 2),
np.sin(angle / 2) * axis[0],
np.sin(angle / 2) * axis[1],
np.sin(angle / 2) * axis[2],
]

t_vals = np.linspace(0, 1, 60)
qs = np.array([slerp(q0, q1, t) for t in t_vals])
qs_nlerp = np.array([nlerp(q0, q1, t) for t in t_vals])

Figura 3D

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.set_box_aspect([1, 1, 1])
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_zlim(-1.5, 1.5)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("slerp (verde) vs nlerp (rojo)")

Eje de rotación

In [ ]:
ax.plot([0, axis[0]], [0, axis[1]], [0, axis[2]], color="gray", linestyle="--", linewidth=1, label="Eje")

arrow_s = ax.plot([], [], [], color=GREEN, linewidth=3)[0]
arrow_n = ax.plot([], [], [], color=RED, linewidth=3)[0]

def animate(frame):
update_arrow(arrow_s, quat_to_rotmat(qs[frame]))
update_arrow(arrow_n, quat_to_rotmat(qs_nlerp[frame]))
return arrow_s, arrow_n

anim = FuncAnimation(fig, animate, frames=len(t_vals), interval=50, blit=False)

In [ ]:
manim.save("slerp_vs_nlerp_3d.html", writer="jshtml")
plt.close(fig)

HTML(filename="slerp_vs_nlerp_3d.html")

## Referencias y para profundizar

Shoemake, Ken. (1985). Animating Rotation with Quaternion Curves. ACM SIGGRAPH Computer Graphics, Vol. 19, No. 3, pp. 245-254.

DOI: 10.1145/325165.325242

El artículo original que introdujo slerp en gráficos por computadora.

Documentación de manim-ce: https://docs.manim.community/en/stable/

Biblioteca de animaciones matemáticas usada en las escenas de este notebook.

scipy.spatial.transform.Rotation: https://docs.scipy.org/doc/scipy/reference/rotations.html

Implementaciones confiables de cuaterniones, slerp y conversiones a matrices de rotación.

Extensiones interesantes:

Splines de cuaterniones (squad) para interpolación $C^1$.
Interpolación en el espacio de Lie $\mathfrak{so}(3)$ mediante logaritmo y exponencial.
Manejo de múltiples keyframes y control de velocidad mediante easing.